# ICU Length-of-Stay Saved Model Demo

This notebook is the public, runnable demo for the ICU length-of-stay project. It uses a committed saved model and synthetic ICU-style rows so reviewers can verify the inference workflow without access to restricted MIMIC-IV patient records.

In [ ]:
from pathlib import Path
import json

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "saved_models" / "short_stay_random_forest"
SAMPLE_PATH = PROJECT_ROOT / "sample_data" / "fake_icu_los_sample.csv"

## Load Model Metadata

The metadata defines the saved model path, target, and expected feature columns.

In [ ]:
with (MODEL_DIR / "metadata.json").open() as f:
    metadata = json.load(f)

metadata

## Prepare Synthetic Rows

The sample CSV contains fake rows only. The mapping below adapts the richer project sample into the compact saved-model schema.

In [ ]:
COLUMN_MAP = {
    "anchor_age": "age",
    "heart_rate_mean": "heart_rate_mean_24h",
    "lab_lactate_max": "lactate_max_24h",
    "rad_note_count_24h": "radiology_note_count_24h",
    "proc_event_count_24h": "procedure_event_count_24h",
    "rx_order_count_24h": "medication_count_24h",
}

sample = pd.read_csv(SAMPLE_PATH)
features = sample.rename(columns=COLUMN_MAP).copy()
features["map_mean_24h"] = (features["sbp_mean"] + 2 * features["dbp_mean"]) / 3
features["abnormal_radiology_flag_24h"] = features[
    [
        "kw_hemorrhage_24h",
        "kw_edema_24h",
        "kw_pneumonia_24h",
        "kw_effusion_24h",
        "kw_stroke_24h",
        "kw_fracture_24h",
        "kw_intubation_24h",
    ]
].fillna(0).max(axis=1)

X = features.reindex(columns=metadata["feature_columns"])
X.head()

## Score Rows With The Saved Pipeline

In [ ]:
model = joblib.load(PROJECT_ROOT / metadata["model_file"])
predictions = model.predict(X)
probabilities = model.predict_proba(X)

results = sample[["demo_stay_id", "anchor_age", "first_careunit", "admission_type"]].copy()
results["predicted_longer_than_2d"] = predictions.astype(int)
results["probability_longer_than_2d"] = probabilities[:, 1].round(3)
results.head(10)

## Data Access Note

The full training pipeline used restricted MIMIC-IV-derived files and is included as scripts in `preprocessing/`, `models/`, and `visualizations/`. Real patient-level rows are intentionally not committed to this public repository.